# Práctica en vivo: tu primer circuito cuántico

**Qiskit Fall Fest 2026** · Natalia Remolina · Universidad Nacional de Colombia · Kat Quantum

Esta práctica sigue el video [**De cero a lo cuántico | Usa una computadora cuántica hoy | Ep. 1**](https://www.youtube.com/watch?v=vSFv_i_FAXg) de IBM Quantum. Si quieres repetirla en casa, el video recorre los mismos pasos.

| Paso | Qué hacemos | En el video |
|---|---|---|
| 0 | Construir el circuito en el **IBM Quantum Composer**, sin código | 3:29 |
| 1 | Preparar el entorno de Python | 2:09 |
| 2 | Escribir el **estado de Bell** en código | 6:21 |
| 3 | Una función que ejecuta circuitos | 7:12 |
| 4 | Ejecutarlo en un **simulador** | 11:19 |
| 5 | Ejecutarlo en un **computador cuántico real** de IBM | 8:28 |
| 6 | Reto para casa: los otros tres estados de Bell | 12:29 |

## Paso 0 · El circuito en el IBM Quantum Composer

Antes de escribir código, construyamos el circuito **arrastrando compuertas**.

1. Abre el [**IBM Quantum Composer**](https://quantum.cloud.ibm.com/composer). No hace falta iniciar sesión; si aparece el aviso de cookies, elige *Required only*.
2. En el menú, haz clic en **Help → Build your first circuit** y sigue el tutorial. O hazlo a mano:
   - Arrastra la compuerta **H** al cable del qubit `q[0]`.
   - Arrastra la **CNOT** (el círculo con una cruz, ⊕): el punto en `q[0]` (control) y el ⊕ en `q[1]` (objetivo).
3. Mira abajo el gráfico **Probabilities**: solo aparecen `00` y `11`, cada uno con 50 %.
4. Mira a la derecha el código **OpenQASM**: el Composer escribe el programa por ti (`h q[0];`, `cx q[0], q[1];`).

**Lo que acabas de construir:**
- La **H (Hadamard)** pone el qubit 0 en **superposición**: un poquito de 0 y un poquito de 1 a la vez.
- La **CNOT** invierte el qubit 1 cuando el qubit 0 vale 1. Juntas crean un **estado de Bell**: los dos qubits quedan **entrelazados** y siempre coinciden.

## Paso 1 · Preparar el entorno

Si usas **Google Colab**, descomenta y ejecuta la primera celda. En un entorno local basta con instalar `requirements.txt` una vez.

In [ ]:
# %pip install -q "qiskit[visualization]" qiskit-aer qiskit-ibm-runtime

In [ ]:
# Las piezas que usaremos en toda la práctica
from qiskit import QuantumCircuit                          # para construir circuitos
from qiskit.visualization import plot_histogram             # para ver los resultados
from qiskit.transpiler import generate_preset_pass_manager  # para adaptar el circuito al hardware
from qiskit_aer import AerSimulator                         # un simulador perfecto
from qiskit_ibm_runtime import QiskitRuntimeService         # la conexión con IBM Quantum
from qiskit_ibm_runtime.executor_sampler import Sampler     # ejecuta el circuito y cuenta resultados

> **Si sigues el video:** Olivia importa `SamplerV2` y `generate_preset_pass_manager` desde `qiskit.transpiler.preset_passmanagers`. Son las mismas herramientas con su nombre anterior; aquí usamos las rutas nuevas de Qiskit para que no aparezcan avisos.

In [ ]:
# Chequeo del entorno: el «cinturón de seguridad» antes de arrancar
import sys
import qiskit, qiskit_aer, qiskit_ibm_runtime

print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)
print("Qiskit IBM Runtime:", qiskit_ibm_runtime.__version__)

## Paso 2 · El estado de Bell en código

Es el mismo circuito que armamos en el Composer, ahora escrito en Python:

In [ ]:
bell = QuantumCircuit(2)   # dos qubits, ambos empiezan en |0⟩
bell.h(0)                  # Hadamard en el qubit 0: superposición
bell.cx(0, 1)              # CNOT: control q0, objetivo q1 → entrelazamiento
bell.measure_all()         # medir los dos qubits (crea el registro clásico «meas»)

bell.draw("mpl")

Este estado se llama $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$, uno de los cuatro **estados de Bell**.

## Paso 3 · Una función para ejecutar circuitos

Esta función hace los pasos que necesita cualquier ejecución, en un simulador o en un computador cuántico real:

1. **Optimizar (transpilar):** reescribir el circuito con las compuertas y conexiones que tiene ese procesador. El resultado se llama *circuito ISA*.
2. **Ejecutar** con el **Sampler**, que repite el circuito `shots` veces.
3. **Post-procesar:** contar cuántas veces salió cada resultado.

In [ ]:
def run_circuit_and_get_counts(circuito, backend, shots=1000):
    # 1. Optimizar: adaptar el circuito al procesador (circuito ISA)
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    circuito_isa = pm.run(circuito)

    # 2. Ejecutar: repetir el circuito `shots` veces
    job = Sampler(mode=backend).run([circuito_isa], shots=shots)
    print("Job ID:", job.job_id())

    # 3. Post-procesar: contar cuántas veces salió cada resultado
    return job.result()[0].data.meas.get_counts()

## Paso 4 · Ejecutarlo en un simulador

Primero lo ejecutamos en **Aer**, un simulador que corre en este computador y no tiene ruido:

In [ ]:
conteos_sim = run_circuit_and_get_counts(bell, AerSimulator())
print(conteos_sim)
plot_histogram(conteos_sim, title="Simulador (perfecto)")

Solo aparecen `00` y `11`, más o menos mitad y mitad. Cada qubit por separado es una moneda, pero **los dos siempre coinciden**: esa es la huella del entrelazamiento.

## Paso 5 · Ejecutarlo en un computador cuántico real

Para usar el hardware de IBM Quantum necesitas una cuenta (el plan Open es gratuito, con el equivalente a 10 minutos de QPU cada 28 días) y tu **clave de API**.

**Guárdala una sola vez** y nunca la compartas: está ligada a tu cuenta personal. Descomenta la celda, pega tu clave, ejecútala y vuelve a comentarla:

In [ ]:
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="PEGA_AQUÍ_TU_CLAVE_DE_API",
#     overwrite=True,
#     set_as_default=True,
# )

Elegimos el procesador **menos ocupado**, para esperar menos en la cola:

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
print("Procesador:", backend.name)

Y lo ejecutamos 1000 veces. El trabajo entra a una **cola** compartida con otras personas, así que puede tardar desde segundos hasta varios minutos:

In [ ]:
conteos_hw = run_circuit_and_get_counts(bell, backend, shots=1000)
print(conteos_hw)

<div class="alert alert-block alert-info">
<b>Si enviaste el trabajo antes</b>

Recupera el resultado con su Job ID, en lugar de esperar la cola. Ejecuta esta celda <b>en lugar de</b> la anterior.
</div>

In [ ]:
# job = service.job("PEGA_AQUÍ_EL_JOB_ID")
# print("Estado:", job.status())
# conteos_hw = job.result()[0].data.meas.get_counts()
# print(conteos_hw)

<div class="alert alert-block alert-info">
<b>Alternativa sin cuenta</b>

Si no tienes cuenta, o el servicio no está disponible, ejecuta esta celda <b>en lugar de</b> las de hardware. <code>FakeTorino</code> es una copia local del procesador <code>ibm_torino</code> (133 qubits) que imita su ruido real.
</div>

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeTorino

backend = FakeTorino()
conteos_hw = run_circuit_and_get_counts(bell, backend, shots=1000)
print(conteos_hw)

Comparemos los dos resultados:

In [ ]:
plot_histogram(
    [conteos_sim, conteos_hw],
    legend=["Simulador (perfecto)", f"{backend.name} (con ruido)"],
    title="Estado de Bell: simulador frente a procesador real",
)

En el procesador real dominan `00` y `11`, pero aparecen **algunos `01` y `10`**. No es un error del circuito: es el **ruido** de un sistema físico. Las compuertas no son perfectas, la lectura a veces se equivoca y la calibración cambia con el tiempo.

**Por qué 1000 shots:** un solo resultado no dice nada; con muchas repeticiones el histograma muestra las probabilidades reales, aun con ruido. Más shots dan más precisión, pero gastan más tiempo de QPU.

¡Acabas de ejecutar tu primer circuito entrelazado en un computador cuántico!

### Comprueba lo que entendiste

<details><summary><b>¿Se puede ver una superposición en una sola medición?</b></summary>

No. Una medición siempre da un valor clásico: 0 o 1. La superposición solo aparece en las estadísticas, después de muchas repeticiones (shots).
</details>

<details><summary><b>¿Por qué el simulador, que es perfecto, no da exactamente 500 veces 00 y 500 veces 11?</b></summary>

Porque simula un proceso aleatorio. Es como lanzar una moneda 1000 veces: aunque la probabilidad sea exactamente 50/50, casi nunca salen justo 500 caras y 500 sellos.
</details>

<details><summary><b>Además del ruido, ¿en qué se diferencia un computador cuántico real de un simulador?</b></summary>

En lo práctico: hay cola de espera, disponibilidad limitada y restricciones propias de cada procesador (sus compuertas y conexiones). Y, sobre todo, a gran escala un simulador no puede imitarlo.
</details>

## Paso 6 · Reto para casa: los otros tres estados de Bell

Hay cuatro estados de Bell. Ya construiste el primero:

| Estado | Fórmula | Resultados al medir |
|---|---|---|
| $\lvert\Phi^+\rangle$ | $(\lvert 00\rangle + \lvert 11\rangle)/\sqrt2$ | 00 y 11 (ya lo hiciste) |
| $\lvert\Phi^-\rangle$ | $(\lvert 00\rangle - \lvert 11\rangle)/\sqrt2$ | 00 y 11 |
| $\lvert\Psi^+\rangle$ | $(\lvert 01\rangle + \lvert 10\rangle)/\sqrt2$ | 01 y 10 |
| $\lvert\Psi^-\rangle$ | $(\lvert 01\rangle - \lvert 10\rangle)/\sqrt2$ | 01 y 10 |

El estado $|\Psi^-\rangle$ se usa más adelante en el curso del video, así que vale la pena resolverlo.

**Pista:** parte del circuito de Bell y agrega **una o dos compuertas de un qubit** (X o Z) al final. Recuerda: X invierte un bit y Z cambia el signo de $|1\rangle$.

Usa esta celda para comprobar tu respuesta: `Statevector` muestra el estado exacto antes de medir.

In [ ]:
from qiskit.quantum_info import Statevector

reto = QuantumCircuit(2)
reto.h(0)
reto.cx(0, 1)
# TODO: agrega aquí una o dos compuertas para obtener otro estado de Bell

Statevector(reto).draw("latex")

## Para seguir

- Video de esta práctica: [De cero a lo cuántico, Ep. 1](https://www.youtube.com/watch?v=vSFv_i_FAXg). El episodio 2 explica la matemática de la superposición y el entrelazamiento.
- [`compuertas_explicadas.ipynb`](../estudio/compuertas_explicadas.ipynb) (material de estudio): cada compuerta en la esfera de Bloch, con experimentos.
- `hello-world_es.ipynb`: el mismo estado de Bell medido con **Estimator** (correlaciones ⟨ZZ⟩ y ⟨XX⟩) y escalado a 100 qubits.